# Example Notebook for Computing MIA

This is a tutorial notebook for quantifying the MIA risk using our KDE-based method. We offer the distance calculations and attack risk calculations as separate classes to offer more flexibility. For example, one may compute the nearest neighbour distances using their own (perhaps more optimized) method, transform the distances into arrays, and finally use the MIA class to compute the risks.

In [ ]:
import numpy as np
import pandas as pd
import cudf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import auc, roc_curve
from NearestNeighbours_GPU import NearestNeighbourDistances
from KDE import KDE_GPU
from MembershipInferenceAttacks import TrueDistributionAttack, RealisticAttack
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def plot_binary_roc(y_true, y_score,  title, xlabel, ylabel, log_scale=False, size=(10,8)):
    """
    Plot ROC curve for binary classification
    
    Args:
        y_true: Ground truth labels (0/1)
        y_score: Predicted probabilities for positive class
    """
    # Calculate ROC curve points
    fpr, tpr, threshold_values = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    
    # Create plot
    plt.figure(figsize=size)
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
                
    # Add labels and styling
    if log_scale == False:
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.subplots_adjust(bottom=0.3)  # Make room for the table
        plt.xlabel(xlabel, fontsize=14)
        plt.ylabel(ylabel, fontsize=14)
        plt.title(title, fontsize=16)
        plt.legend(loc="lower right", fontsize=12)
        plt.grid(True, alpha=0.5)

    else:
        plt.xscale('log')  # Log scale for x-axis
        plt.yscale('log')  # Log scale for y-axis
        plt.xlim(0.000001, 1)  # Avoid log(0) issues
        plt.ylim(0.000001, 1)
        plt.xlabel(xlabel, fontsize=14)
        plt.ylabel(ylabel, fontsize=14)
        plt.title(title, fontsize=16)
        plt.legend(loc="lower right", fontsize=12)
        plt.grid(True, which="both", ls="--")

    
    return plt


## Loading the Datasets

In [ ]:
# For optimized NN-distance calculations, load the original datasets as CuDF dataframes
real_df = cudf.read_csv("real_df.csv")
synthetic_df = cudf.read_csv("synthetic_df.csv")
unseen_df = cudf.read_csv("holdout_df.csv")

In [ ]:
print("Real (training) data shape:", real_df.shape)
print("Synthetic data shape:", synthetic_df.shape)
print("Unseen/holdout data shape:", unseen_df.shape)
print("Number of features:", len(real_df.columns))

## Computing the Nearest Neighbour Distances (real-to-synthetic and unseen-to-synthetic)

In [ ]:
real_to_synthetic = NearestNeighbourDistances(
    real_df=real_df,
    synthetic_df=synthetic_df,
    categorical_columns=[True, True, False, False, True, True, True, True, False, False], # an example
    unique_threshold=None, # If categorical_columns = None, provide an integer threshold to determine the categorical columns
    num_nearest=1,
    normalize='minmax',
    x_chunk_size=30000, # Reduce this if you get GPU memory error
    y_chunk_size=100000,
    rmm_pool_size='14.5GB'
)

unseen_to_synthetic = NearestNeighbourDistances(
    real_df=unseen_df,
    synthetic_df=synthetic_df,
    categorical_columns=[True, True, False, False, True, True, True, True, False, False],
    unique_threshold=None,
    num_nearest=1,
    normalize='minmax',
    x_chunk_size=30000,
    y_chunk_size=100000,
    rmm_pool_size='14.5GB'
)

In [ ]:
%%time

# Get the distance dataframes
real_to_synth_dists_df = real_to_synthetic.compute("real_to_synth_distances.csv")
real_distances = np.asarray(real_to_synth_dists_df["nearest_1"]) # true member distances

In [ ]:
%%time

unseen_to_synth_dists_df = unseen_to_synthetic.compute("unseen_to_synth_distances.csv")
holdout_distances = np.asarray(unseen_to_synth_dists_df["nearest_1"]) # true non-member distances

## True Distribution Attack

In [ ]:
%%time

td_mia = TrueDistributionAttack(
    real_distances,
    holdout_distances,
    partition_param=1, # use all the real distances for calculations
    split_param=0.3, # use a 70-30% split for train and test sets
    equal=True, # keep an equal number of members and non-members in the test set
    device='gpu' # KDE fitting and evaluations are done on GPU
)

td_mia.fit_kdes(kernel='gaussian', rule='scott')
acc, f1 = td_mia.evaluate(batch_size=700) # increase batch_size if your GPU has more memory
print("True distribution attack accuracy:", acc)
print("True distribution attack F1 score:", f1)

# Get labels and probabilities for plotting ROC curve
y_true = td_mia.true_membership_labels_
y_pred_proba = td_mia.membership_probas_
plot_binary_roc(y_true, y_pred_proba, xlabel='False Positive Rate (1 - Specificity)', ylabel='True Positive Rate (Sensitivity)', title='Log ROC Curve: True Distribution Attack', log_scale=True, size=(6,6))


## Realistic Attack

In [ ]:
%%time

realistic_mia = RealisticAttack(
    real_distances,
    holdout_distances,
    partition_param=1,
    split_param=0.3,
    equal=True,
    device='gpu',
    threshold_percentile=50
)

realistic_mia.fit_kdes(kernel='gaussian', rule='scott')
acc, f1 = realistic_mia.evaluate(batch_size=700)
print("Realistic attack accuracy at 50%ile threshold:", acc)
print("Realistic attack F1 score at 50%ile threshold:", f1)

# Get labels and probabilities for plotting ROC curve
y_true = realistic_mia.true_membership_labels_
y_pred_proba = realistic_mia.membership_probas_
plot_binary_roc(y_true, y_pred_proba, xlabel='False Positive Rate (1 - Specificity)', ylabel='True Positive Rate (Sensitivity)', title='Log ROC Curve: Realistic Attack', log_scale=True, size=(6,6))


In [ ]:
%%time

realistic_mia = RealisticAttack(
    real_distances,
    holdout_distances,
    partition_param=1,
    split_param=0.3,
    equal=True,
    device='gpu',
    threshold_percentile=60
)

realistic_mia.fit_kdes(kernel='gaussian', rule='scott')
acc, f1 = realistic_mia.evaluate(batch_size=700)
print("Realistic attack accuracy at 60%ile threshold:", acc)
print("Realistic attack F1 score at 60%ile threshold:", f1)

# Get labels and probabilities for plotting ROC curve
y_true = realistic_mia.true_membership_labels_
y_pred_proba = realistic_mia.membership_probas_
plot_binary_roc(y_true, y_pred_proba, xlabel='False Positive Rate (1 - Specificity)', ylabel='True Positive Rate (Sensitivity)', title='Log ROC Curve: Realistic Attack', log_scale=True, size=(6,6))
